# Qwen3.5-27B Post-LoRA Torque-Control Stress Test

上一轮 blind causal validation 的六个预设 gate 通过了五个：LoRA 的盲测行为改进、局部层窗贡献、L50 state 的充分性与必要性，以及 L42 torque direction 到 L50 outcome probe 的传播均成立；唯一失败的是 L42 torque direction 对最终答案的直接控制。

[实验设置与运行说明](../../../docs/experiment_notes/07_appendix_lora_training/790dbceec930c6f8.md)


In [ ]:
# Run once. Every later code cell reports progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 15 tracked cells.')


## 1. Freeze the follow-up protocol

The prior blind run is treated as immutable input.


In [ ]:
%%cell_progress 1/15 Mount Drive and select a complete VisualTorque project
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Auditing candidate project roots')
from pathlib import Path
import shutil

PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
REQUIRED_TOOLKIT_PATHS = (
    'pyproject.toml',
    'src/visual_torque_toolkit/__init__.py',
    'src/visual_torque_toolkit/contracts.py',
    'src/visual_torque_toolkit/classify.py',
    'src/visual_torque_toolkit/dataset.py',
    'src/visual_torque_toolkit/edits.py',
    'src/visual_torque_toolkit/generate.py',
    'src/visual_torque_toolkit/prompts.py',
    'src/visual_torque_toolkit/render.py',
    'src/visual_torque_toolkit/splits.py',
    'src/visual_torque_toolkit/world.py',
    'src/visual_torque_toolkit/behavior/__init__.py',
    'src/visual_torque_toolkit/behavior/adapter.py',
    'src/visual_torque_toolkit/behavior/data.py',
    'src/visual_torque_toolkit/behavior/schema.py',
)

def missing_toolkit_paths(project_root):
    toolkit_root = project_root / 'toolkit'
    return [
        relative for relative in REQUIRED_TOOLKIT_PATHS
        if not (toolkit_root / relative).is_file()
    ]

candidate_audit = {
    str(candidate): missing_toolkit_paths(candidate)
    for candidate in PROJECT_CANDIDATES
}
complete_candidates = [
    candidate for candidate in PROJECT_CANDIDATES
    if not candidate_audit[str(candidate)]
]
assert complete_candidates, (
    'No complete VisualTorque toolkit was found. '
    f'Missing-file audit: {candidate_audit}'
)
PROJECT_ROOT = complete_candidates[0]

cell_stage('Copying immutable toolkit source to local SSD')
DRIVE_TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
TOOLKIT_ROOT = Path('/content/visualtorque_toolkit_runtime')
if TOOLKIT_ROOT.exists():
    shutil.rmtree(TOOLKIT_ROOT)
shutil.copytree(DRIVE_TOOLKIT_ROOT, TOOLKIT_ROOT)
print('Project:', PROJECT_ROOT)
print('Toolkit runtime:', TOOLKIT_ROOT)


In [ ]:
%%cell_progress 2/15 Install and verify the frozen causal environment
import importlib
import importlib.metadata
import json
import os
import subprocess
import sys

cell_stage('Configuring caches before importing Torch')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('HF_HOME', '/content/hf_cache')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

cell_stage('Removing incompatible optional Torch packages')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio', 'torchao'],
    check=False,
)
importlib.invalidate_caches()

cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    '--upgrade-strategy', 'only-if-needed',
    'transformers==5.14.0', 'peft==0.20.0', 'accelerate>=1.10',
    'sentencepiece', 'protobuf', 'huggingface_hub>=0.34',
    'safetensors', 'tqdm>=4.66', 'scikit-learn>=1.5',
    'scipy>=1.13', 'pandas>=2.2', 'matplotlib>=3.8',
    'joblib>=1.4', 'pillow>=10.4',
])

cell_stage('Importing the immutable toolkit copy')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if module_name == 'visual_torque_toolkit' or module_name.startswith('visual_torque_toolkit.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import visual_torque_toolkit
from IPython.display import display
from peft import PeftModel
from scipy.stats import binomtest
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter
from visual_torque_toolkit.generate import (
    GenerationConfig, generate_multi_stack_scenes,
)
from visual_torque_toolkit.prompts import QuestionKind, generate_questions
from visual_torque_toolkit.render import RenderStyle, render_scene

# Defensive Qwen3.5 compatibility patch for exact candidate scoring.
def append_text_prefix_aligned(self, inputs, prefix):
    if not prefix:
        return inputs
    batch_size = inputs['input_ids'].shape[0]
    prefix_tensor = torch.tensor(
        prefix,
        dtype=inputs['input_ids'].dtype,
        device=inputs['input_ids'].device,
    ).unsqueeze(0).expand(batch_size, -1)
    inputs['input_ids'] = torch.cat((inputs['input_ids'], prefix_tensor), dim=1)
    attention_extension = torch.ones(
        (batch_size, len(prefix)),
        dtype=inputs['attention_mask'].dtype,
        device=inputs['attention_mask'].device,
    )
    inputs['attention_mask'] = torch.cat(
        (inputs['attention_mask'], attention_extension), dim=1
    )
    for key in ('token_type_ids', 'mm_token_type_ids'):
        if key not in inputs:
            continue
        extension = torch.zeros(
            (batch_size, len(prefix)),
            dtype=inputs[key].dtype,
            device=inputs[key].device,
        )
        inputs[key] = torch.cat((inputs[key], extension), dim=1)
    inputs.pop('position_ids', None)
    return inputs

HuggingFaceMultimodalAdapter._append_text_prefix = append_text_prefix_aligned
torch.backends.cuda.matmul.allow_tf32 = True
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents
try:
    torchao_version = importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
assert torchao_version is None
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('transformers:', importlib.metadata.version('transformers'))
print('peft:', importlib.metadata.version('peft'))
print('Toolkit:', toolkit_path)


In [ ]:
%%cell_progress 3/15 Lock inputs, the two-stage split, and all decision rules
cell_stage('Verifying prior runs and immutable identities')
import hashlib
from collections import Counter, defaultdict
from datetime import datetime, timezone
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

assert torch.cuda.is_available(), 'Select a GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
assert GPU_MEMORY_GIB >= 88, (GPU_NAME, GPU_MEMORY_GIB)

MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
EXPECTED_LAYER_COUNT = 64
PHYSICAL_LAYERS = (38, 42, 46)
OUTCOME_LAYER = 50
TOKEN_GROUPS = ('final_token', 'visual_tokens', 'all_attended_tokens')
PHYSICAL_AXIS_NAMES = ('net_torque_rank1', 'side_torque_rank2')
DEV_DOSES = (-4.0, -2.0, 0.0, 2.0, 4.0)
TEST_DOSES = (-8.0, -4.0, -2.0, -1.0, 0.0, 1.0, 2.0, 4.0, 8.0)
PRIMARY_TEST_DOSES = (-4.0, -2.0, -1.0, 0.0, 1.0, 2.0, 4.0)
RANDOM_DOSES = (-4.0, 0.0, 4.0)
POSITIVE_CONTROL_DOSES = (-4.0, -2.0, 0.0, 2.0, 4.0)
RANDOM_CONTROL_COUNT = 32
INTERVENTION_BATCH_SIZE = 3
BOOTSTRAP_REPLICATES = 10000
SIGN_FLIP_REPLICATES = 10000
ALPHA_LEVEL = 0.05
SEED = 20260906
RUN_VERSION = 'v1'

TRAIN_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_bf16_lora_weight_shortcut_v1'
ADAPTER_DIR = TRAIN_ROOT / 'adapter_final'
ADAPTER_MANIFEST_PATH = TRAIN_ROOT / 'ADAPTER_MANIFEST.json'
DISCOVERY_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_base_vs_lora_mechanism_discovery_v1'
DISCOVERY_COMPLETE_PATH = DISCOVERY_ROOT / 'RUN_COMPLETE.json'
FROZEN_HYPOTHESES_PATH = DISCOVERY_ROOT / 'FROZEN_MECHANISM_HYPOTHESES.json'
LORA_CAPTURE_DIR = DISCOVERY_ROOT / 'lora_capture'
BLIND_ROOT = PROJECT_ROOT / 'results' / 'qwen27b_postlora_blind_causal_validation_v1'
BLIND_COMPLETE_PATH = BLIND_ROOT / 'RUN_COMPLETE.json'
BLIND_DECISION_PATH = BLIND_ROOT / 'analysis' / 'causal_decision.json'
BLIND_DATASET_DIR = BLIND_ROOT / 'blind_dataset'
PRIOR_CAUSAL_MANIFEST_PATH = BLIND_ROOT / 'causal_example_manifest.jsonl'
BLIND_SCENE_MANIFEST_PATH = BLIND_ROOT / 'blind_scene_manifest.jsonl'

for required in (
    ADAPTER_DIR / 'adapter_config.json', ADAPTER_MANIFEST_PATH,
    DISCOVERY_COMPLETE_PATH, FROZEN_HYPOTHESES_PATH,
    LORA_CAPTURE_DIR / 'capture_config.json',
    BLIND_COMPLETE_PATH, BLIND_DECISION_PATH,
    BLIND_DATASET_DIR / 'scenes.jsonl', BLIND_DATASET_DIR / 'questions.jsonl',
    PRIOR_CAUSAL_MANIFEST_PATH, BLIND_SCENE_MANIFEST_PATH,
):
    assert required.is_file(), required

capture_shard_paths = sorted(LORA_CAPTURE_DIR.glob('shard_*.pt'))
assert capture_shard_paths, LORA_CAPTURE_DIR

def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if torch.is_tensor(value) and value.numel() == 1:
        return value.item()
    raise TypeError(f'Cannot serialize {type(value).__name__}')

def write_json_atomic(file_path, value):
    file_path = Path(file_path)
    temporary = file_path.with_suffix(file_path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(
            value, indent=2, ensure_ascii=False, sort_keys=True,
            default=json_default,
        ) + '\n',
        encoding='utf-8',
    )
    temporary.replace(file_path)

def read_jsonl(file_path):
    with Path(file_path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def sha256_file(file_path):
    digest = hashlib.sha256()
    with Path(file_path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value!r}'.encode('utf-8')).hexdigest()

adapter_manifest = json.loads(ADAPTER_MANIFEST_PATH.read_text(encoding='utf-8'))
frozen_hypotheses = json.loads(FROZEN_HYPOTHESES_PATH.read_text(encoding='utf-8'))
blind_decision = json.loads(BLIND_DECISION_PATH.read_text(encoding='utf-8'))
assert adapter_manifest['base_model_id'] == MODEL_ID
assert adapter_manifest['base_model_revision'] == MODEL_REVISION
assert frozen_hypotheses['model_id'] == MODEL_ID
assert frozen_hypotheses['model_revision'] == MODEL_REVISION
assert int(frozen_hypotheses['lora_net_torque_layer']['layer_index']) == 42
assert int(frozen_hypotheses['lora_outcome_layer']['layer_index']) == OUTCOME_LAYER
assert not frozen_hypotheses['probe_test_used_for_layer_selection']
assert blind_decision['gates']['torque_direction_controls_output'] is False

blind_rows = read_jsonl(BLIND_SCENE_MANIFEST_PATH)
prior_causal_rows = read_jsonl(PRIOR_CAUSAL_MANIFEST_PATH)
strict_conflict_ids = {
    row['scene_id'] for row in blind_rows
    if row['weight_cue'] in {'left', 'right'}
    and row['outcome'] in {'left', 'right'}
    and row['weight_cue'] != row['outcome']
}
route_dev_scene_ids = {row['scene_id'] for row in prior_causal_rows}
route_test_scene_ids = strict_conflict_ids - route_dev_scene_ids
assert len(strict_conflict_ids) == 48
assert len(route_dev_scene_ids) == len(route_test_scene_ids) == 24
assert not route_dev_scene_ids & route_test_scene_ids

OUTPUT_ROOT = (
    PROJECT_ROOT / 'results'
    / f'qwen27b_postlora_torque_control_stresstest_{RUN_VERSION}'
)
DEV_DIR = OUTPUT_ROOT / 'route_development'
TEST_DIR = OUTPUT_ROOT / 'intervention_confirmatory'
POSITIVE_CONTROL_DIR = OUTPUT_ROOT / 'l50_positive_control'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for directory in (
    OUTPUT_ROOT, DEV_DIR, TEST_DIR, POSITIVE_CONTROL_DIR,
    ANALYSIS_DIR, FIGURE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

protocol_identity = {
    'schema_version': 'qwen27b-postlora-torque-control-stresstest-1.0',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'adapter_manifest_sha256': sha256_file(ADAPTER_MANIFEST_PATH),
    'frozen_hypotheses_sha256': sha256_file(FROZEN_HYPOTHESES_PATH),
    'lora_capture_config_sha256': sha256_file(
        LORA_CAPTURE_DIR / 'capture_config.json'
    ),
    'lora_capture_shard_hashes': {
        path.name: sha256_file(path) for path in capture_shard_paths
    },
    'blind_complete_sha256': sha256_file(BLIND_COMPLETE_PATH),
    'blind_decision_sha256': sha256_file(BLIND_DECISION_PATH),
    'blind_scene_manifest_sha256': sha256_file(BLIND_SCENE_MANIFEST_PATH),
    'prior_causal_manifest_sha256': sha256_file(PRIOR_CAUSAL_MANIFEST_PATH),
    'physical_layers': list(PHYSICAL_LAYERS),
    'outcome_layer': OUTCOME_LAYER,
    'token_groups': list(TOKEN_GROUPS),
    'physical_axis_names': list(PHYSICAL_AXIS_NAMES),
    'dev_doses': list(DEV_DOSES),
    'test_doses': list(TEST_DOSES),
    'primary_test_doses': list(PRIMARY_TEST_DOSES),
    'random_doses': list(RANDOM_DOSES),
    'positive_control_doses': list(POSITIVE_CONTROL_DOSES),
    'random_control_count': RANDOM_CONTROL_COUNT,
    'intervention_batch_size': INTERVENTION_BATCH_SIZE,
    'route_dev_scene_ids': sorted(route_dev_scene_ids),
    'route_test_scene_ids': sorted(route_test_scene_ids),
    'selection_rule': (
        'maximize route-dev scene-mean left-minus-right output slope; '
        'break ties by mediator slope, then deterministic candidate name'
    ),
    'confirmatory_unit': 'physical scene after A/B/C permutation averaging',
    'random_control_rule': (
        '32 seeded orthonormal directions orthogonal to both fitted physical '
        'axes at the frozen selected layer'
    ),
    'primary_gate_rule': (
        'positive scene-bootstrap output slope, positive +/-4 contrast, '
        'one-sided empirical p<=.05 versus random directions, and positive '
        'slope for every A/B/C gold-label stratum'
    ),
    'seed': SEED,
}
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'frozen_protocol.json'
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing protocol differs. Increment RUN_VERSION.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

print('GPU:', GPU_NAME, '| memory GiB:', round(GPU_MEMORY_GIB, 1))
print('Route-development scenes:', len(route_dev_scene_ids))
print('Intervention-confirmatory scenes:', len(route_test_scene_ids))
print('Output:', OUTPUT_ROOT)


In [ ]:
%%cell_progress 4/15 Fit frozen layer-specific physical axes from probe-train only
cell_stage('Loading sparse LoRA discovery captures')
capture_config = json.loads(
    (LORA_CAPTURE_DIR / 'capture_config.json').read_text(encoding='utf-8')
)['identity']
captured_layers = [int(value) for value in capture_config['layer_indices']]
for layer_index in (*PHYSICAL_LAYERS, OUTCOME_LAYER):
    assert layer_index in captured_layers, (layer_index, captured_layers)
layer_position = {
    layer_index: captured_layers.index(layer_index)
    for layer_index in (*PHYSICAL_LAYERS, OUTCOME_LAYER)
}

activation_by_id = {}
record_by_id = {}
for file_path in tqdm(
    sorted(LORA_CAPTURE_DIR.glob('shard_*.pt')),
    desc='Loading LoRA discovery shards', unit='shard',
):
    payload = torch.load(file_path, map_location='cpu', weights_only=False)
    activations = payload['activations'].float().numpy()
    for row_index, record in enumerate(payload['records']):
        if 'main' not in str(record['role']):
            continue
        example_id = record['example_id']
        assert example_id not in activation_by_id
        activation_by_id[example_id] = {
            layer: activations[row_index, position]
            for layer, position in layer_position.items()
        }
        record_by_id[example_id] = record

main_ids = sorted(activation_by_id)
metadata = pd.DataFrame([record_by_id[value] for value in main_ids])
assert metadata['scene_id'].is_unique
assert set(metadata['split']) == {'probe_train', 'probe_dev'}
train_mask = metadata['split'].eq('probe_train').to_numpy()
dev_mask = metadata['split'].eq('probe_dev').to_numpy()
assert not metadata.loc[dev_mask, 'seen_in_sft'].any()

axis_payload = {
    'schema_version': 'qwen27b-postlora-layer-physical-axes-1.0',
    'fit_split': 'probe_train',
    'calibration_split': 'probe_dev',
    'ridge_alpha': 100.0,
    'directions': {},
    'metrics': [],
}

def unit_vector(value):
    value = np.asarray(value, dtype=np.float32)
    norm = float(np.linalg.norm(value))
    assert np.isfinite(norm) and norm > 0
    return value / norm

for layer_index in PHYSICAL_LAYERS:
    X = np.stack([
        activation_by_id[value][layer_index] for value in main_ids
    ]).astype(np.float32)
    X_train, X_dev = X[train_mask], X[dev_mask]
    scaler = StandardScaler().fit(X_train)
    Z_train = scaler.transform(X_train)
    Z_dev = scaler.transform(X_dev)

    y_net = metadata['net_torque'].to_numpy(dtype=np.float32)
    net_model = Ridge(alpha=100.0, solver='lsqr', max_iter=5000).fit(
        Z_train, y_net[train_mask]
    )
    net_raw = np.asarray(net_model.coef_, dtype=np.float32) / np.asarray(
        scaler.scale_, dtype=np.float32
    )
    net_direction = unit_vector(net_raw)
    if np.corrcoef(X_dev @ net_direction, y_net[dev_mask])[0, 1] < 0:
        net_direction *= -1

    y_side = metadata[['left_torque', 'right_torque']].to_numpy(dtype=np.float32)
    side_model = Ridge(alpha=100.0, solver='lsqr', max_iter=5000).fit(
        Z_train, y_side[train_mask]
    )
    raw_map = np.asarray(side_model.coef_, dtype=np.float32).T / np.asarray(
        scaler.scale_, dtype=np.float32
    )[:, None]
    target_change = np.asarray([1.0, -1.0], dtype=np.float32)
    side_direction = unit_vector(
        raw_map @ np.linalg.pinv(raw_map.T @ raw_map) @ target_change
    )
    if np.corrcoef(X_dev @ side_direction, y_net[dev_mask])[0, 1] < 0:
        side_direction *= -1

    directions = {
        'net_torque_rank1': net_direction,
        'side_torque_rank2': side_direction,
    }
    axis_payload['directions'][str(layer_index)] = {}
    for axis_name, direction in directions.items():
        projection = X_dev @ direction
        step = float(np.std(projection, ddof=1))
        assert np.isfinite(step) and step > 0
        axis_payload['directions'][str(layer_index)][axis_name] = {
            'vector': torch.from_numpy(direction),
            'step': step,
        }
        axis_payload['metrics'].append({
            'layer_index': layer_index,
            'axis_name': axis_name,
            'dev_projection_net_torque_r': float(
                np.corrcoef(projection, y_net[dev_mask])[0, 1]
            ),
            'step_projection_sd': step,
            'net_torque_probe_dev_r2': float(r2_score(
                y_net[dev_mask], net_model.predict(Z_dev)
            )),
        })

cell_stage('Loading the frozen L50 outcome direction')
probe_records = {
    row['target']: row for row in frozen_hypotheses['saved_probes']
}
outcome_probe_path = Path(probe_records['outcome']['path'])
assert sha256_file(outcome_probe_path) == probe_records['outcome']['sha256']
outcome_probe = joblib.load(outcome_probe_path)
outcome_scaler = outcome_probe.named_steps['standardscaler']
outcome_classifier = outcome_probe.named_steps['ridgeclassifier']
outcome_classes = [str(value) for value in outcome_classifier.classes_.tolist()]
left_index = outcome_classes.index('left')
right_index = outcome_classes.index('right')
outcome_raw = (
    np.asarray(outcome_classifier.coef_[left_index], dtype=np.float32)
    - np.asarray(outcome_classifier.coef_[right_index], dtype=np.float32)
) / np.asarray(outcome_scaler.scale_, dtype=np.float32)
outcome_direction = unit_vector(outcome_raw)
X50 = np.stack([
    activation_by_id[value][OUTCOME_LAYER] for value in main_ids
]).astype(np.float32)
outcome_step = float(np.std(X50[dev_mask] @ outcome_direction, ddof=1))
assert outcome_step > 0
axis_payload['outcome_direction'] = {
    'layer_index': OUTCOME_LAYER,
    'vector': torch.from_numpy(outcome_direction),
    'step': outcome_step,
    'classes': outcome_classes,
    'source_probe_sha256': probe_records['outcome']['sha256'],
}

AXIS_PATH = OUTPUT_ROOT / 'frozen_intervention_axes.pt'
if AXIS_PATH.is_file():
    previous_axes = torch.load(AXIS_PATH, map_location='cpu', weights_only=False)
    assert previous_axes['schema_version'] == axis_payload['schema_version']
    for layer_index in PHYSICAL_LAYERS:
        for axis_name in PHYSICAL_AXIS_NAMES:
            assert torch.allclose(
                previous_axes['directions'][str(layer_index)][axis_name]['vector'],
                axis_payload['directions'][str(layer_index)][axis_name]['vector'],
                atol=1e-7, rtol=0,
            )
else:
    torch.save(axis_payload, AXIS_PATH)

axis_metrics = pd.DataFrame(axis_payload['metrics'])
axis_metrics.to_csv(ANALYSIS_DIR / 'frozen_axis_metrics.csv', index=False)
display(axis_metrics.round(4))
print('L50 positive-control step:', outcome_step)


In [ ]:
%%cell_progress 5/15 Build label-balanced development and confirmatory manifests
cell_stage('Loading all six blind permutations without scoring the model')
blind_config = BehaviorRunConfig(
    dataset_dir=BLIND_DATASET_DIR,
    output_dir=OUTPUT_ROOT / '_unused_behavior_config',
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('blind_confirmatory',),
    batch_size=1,
    seed=SEED,
)
all_blind_examples = list(load_behavior_examples(blind_config))
examples_by_scene = defaultdict(list)
for example in all_blind_examples:
    examples_by_scene[example.scene_id].append(example)
assert all(len(values) == 6 for values in examples_by_scene.values())

def choose_one_per_gold(scene_id):
    rows = examples_by_scene[scene_id]
    chosen = {}
    for gold_label in ('A', 'B', 'C'):
        candidates = [row for row in rows if row.gold_label == gold_label]
        assert len(candidates) == 2
        chosen[gold_label] = sorted(
            candidates, key=lambda row: stable_key(row.example_id)
        )[0]
    return chosen

route_dev_examples = []
route_dev_manifest = []
for index, scene_id in enumerate(sorted(route_dev_scene_ids, key=stable_key)):
    desired_label = ('A', 'B', 'C')[index % 3]
    example = choose_one_per_gold(scene_id)[desired_label]
    route_dev_examples.append(example)
    route_dev_manifest.append({
        'role': 'route_development',
        'scene_id': scene_id,
        'example_id': example.example_id,
        'gold_label': example.gold_label,
        'gold_semantic': str(example.gold_semantic),
    })

route_test_examples = []
route_test_manifest = []
for scene_id in sorted(route_test_scene_ids, key=stable_key):
    for gold_label, example in choose_one_per_gold(scene_id).items():
        route_test_examples.append(example)
        route_test_manifest.append({
            'role': 'intervention_confirmatory',
            'scene_id': scene_id,
            'example_id': example.example_id,
            'gold_label': gold_label,
            'gold_semantic': str(example.gold_semantic),
        })

assert len(route_dev_examples) == 24
assert len(route_test_examples) == 72
assert len({row.scene_id for row in route_test_examples}) == 24
assert Counter(row.gold_label for row in route_dev_examples) == Counter(
    {'A': 8, 'B': 8, 'C': 8}
)
assert Counter(row.gold_label for row in route_test_examples) == Counter(
    {'A': 24, 'B': 24, 'C': 24}
)

DEV_MANIFEST_PATH = OUTPUT_ROOT / 'route_development_manifest.json'
TEST_MANIFEST_PATH = OUTPUT_ROOT / 'intervention_confirmatory_manifest.json'
write_json_atomic(DEV_MANIFEST_PATH, route_dev_manifest)
write_json_atomic(TEST_MANIFEST_PATH, route_test_manifest)
print('Development prompts:', len(route_dev_examples))
print('Confirmatory prompts:', len(route_test_examples))
display(pd.crosstab(
    pd.DataFrame(route_test_manifest)['gold_label'],
    pd.DataFrame(route_test_manifest)['gold_semantic'],
))


## 2. Load the frozen LoRA model and define interventions

Every direction is fitted from the earlier probe-train activations.


In [ ]:
%%cell_progress 6/15 Load the pinned base model and verified LoRA adapter
cell_stage('Loading the pinned BF16 base checkpoint')
adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
    device_map='cuda:0',
)
adapter.model.eval()
assert not hasattr(adapter.model, 'peft_config')

cell_stage('Verifying every saved adapter file')
def hash_tree(directory):
    return {
        file_path.relative_to(directory).as_posix(): sha256_file(file_path)
        for file_path in sorted(Path(directory).rglob('*'))
        if file_path.is_file()
    }

assert hash_tree(ADAPTER_DIR) == adapter_manifest['adapter_files'], (
    'Saved LoRA files do not match ADAPTER_MANIFEST.json.'
)

cell_stage('Attaching LoRA read-only')
adapter.model = PeftModel.from_pretrained(
    adapter.model,
    str(ADAPTER_DIR),
    is_trainable=False,
)
adapter.model.eval()
for parameter in adapter.model.parameters():
    parameter.requires_grad_(False)
assert not any(parameter.requires_grad for parameter in adapter.model.parameters())

def discover_decoder_layers(model):
    candidates = [
        (name, module)
        for name, module in model.named_modules()
        if (
            isinstance(module, torch.nn.ModuleList)
            and len(module) == EXPECTED_LAYER_COUNT
            and name.endswith('language_model.layers')
        )
    ]
    if not candidates:
        candidates = [
            (name, module)
            for name, module in model.named_modules()
            if isinstance(module, torch.nn.ModuleList)
            and len(module) == EXPECTED_LAYER_COUNT
        ]
    assert candidates, 'Could not locate the 64 decoder layers.'
    candidates.sort(key=lambda item: (item[0].count('.'), len(item[0])))
    return candidates[0]

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
print('Model class:', type(adapter.model).__name__)
print('Decoder path:', LAYER_PATH)
print('Adapter verification: passed')


In [ ]:
%%cell_progress 7/15 Define token-group intervention and exact semantic scoring
cell_stage('Defining hook-safe batched intervention utilities')
import math

scene_metadata = {row['scene_id']: row for row in blind_rows}

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        tensor = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        tensor = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        tensor = output.last_hidden_state
    else:
        raise TypeError(type(output).__name__)
    assert tensor.ndim == 3, tuple(tensor.shape)
    return tensor

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def chunks(values, size):
    values = list(values)
    for start in range(0, len(values), size):
        yield values[start:start + size]

def prepare_repeated(example, repeats):
    plan = adapter._candidate_plan(example)
    if plan is None:
        raise RuntimeError('A/B/C must be exact single-token candidates.')
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for _ in range(repeats)],
            add_generation_prompt=True,
        )),
        fixed_prefix,
    )
    attention = inputs['attention_mask'].bool()
    positions = (
        attention.long()
        * torch.arange(attention.shape[1], device=attention.device).unsqueeze(0)
    ).max(dim=1).values
    return inputs, positions, token_by_label

def make_token_mask(inputs, positions, token_group):
    attention = inputs['attention_mask'].bool()
    rows = torch.arange(attention.shape[0], device=attention.device)
    final = torch.zeros_like(attention)
    final[rows, positions] = True
    if token_group == 'final_token':
        selected = final
    else:
        assert 'mm_token_type_ids' in inputs, (
            'Qwen3.5 mm_token_type_ids is required for token-group tests.'
        )
        mm_types = inputs['mm_token_type_ids']
        assert mm_types.shape == attention.shape, (
            mm_types.shape, attention.shape
        )
        visual = attention & mm_types.ne(0)
        assert visual.any(dim=1).all(), 'No visual tokens were detected.'
        if token_group == 'visual_tokens':
            selected = visual
        elif token_group == 'all_attended_tokens':
            selected = attention
        else:
            raise ValueError(token_group)
    assert selected.any(dim=1).all()
    return selected

def outcome_probe_margins(states):
    scores = np.asarray(outcome_probe.decision_function(
        states.detach().cpu().float().numpy()
    ))
    if scores.ndim == 1:
        scores = scores[None, :]
    return scores[:, left_index] - scores[:, right_index]

def score_rows(example, logits, positions, token_by_label):
    rows = torch.arange(logits.shape[0], device=logits.device)
    logprobs = torch.log_softmax(logits[rows, positions, :].float(), dim=-1)
    results = []
    for row_index in range(logits.shape[0]):
        semantic_scores = {
            str(example.label_to_semantic[label]): float(
                logprobs[row_index, token_by_label[label]].detach().cpu()
            )
            for label in example.labels
        }
        prediction = max(semantic_scores, key=semantic_scores.get)
        outcome = str(example.gold_semantic)
        direction_sign = 1.0 if outcome == 'left' else -1.0
        results.append({
            'scene_id': example.scene_id,
            'example_id': example.example_id,
            'variant': example.variant,
            'gold_label': example.gold_label,
            'outcome': outcome,
            'weight_cue': scene_metadata[example.scene_id]['weight_cue'],
            'style': scene_metadata[example.scene_id]['style'],
            'predicted_semantic': prediction,
            'correct': prediction == outcome,
            'semantic_logprobs': semantic_scores,
            'left_right_margin': float(
                semantic_scores['left'] - semantic_scores['right']
            ),
            'correct_margin': float(direction_sign * (
                semantic_scores['left'] - semantic_scores['right']
            )),
        })
    return results

def forward_intervention_batch(
    example, layer_index, direction, step, doses, token_group,
    axis_name, capture_l50=True,
):
    doses = [float(value) for value in doses]
    inputs, positions, token_by_label = prepare_repeated(example, len(doses))
    token_mask = make_token_mask(inputs, positions, token_group)
    rows = torch.arange(len(doses), device=positions.device)
    direction = direction.detach().cpu().float()
    mediator = {}

    def intervention_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        modified = hidden.clone()
        unit = direction.to(hidden.device, hidden.dtype)
        dose_tensor = torch.tensor(
            doses, device=hidden.device, dtype=hidden.dtype
        ) * float(step)
        token_weights = token_mask.to(hidden.dtype)
        token_weights = token_weights / torch.sqrt(
            token_weights.sum(dim=1, keepdim=True).clamp_min(1.0)
        )
        modified += (
            token_weights.unsqueeze(-1)
            * dose_tensor[:, None, None]
            * unit[None, None, :]
        )
        return replace_output_hidden(output, modified)

    handles = [
        DECODER_LAYERS[layer_index].register_forward_hook(intervention_hook)
    ]
    if capture_l50 and layer_index < OUTCOME_LAYER:
        def mediator_hook(_module, _inputs, output):
            hidden = output_hidden_tensor(output)
            mediator['states'] = hidden[
                rows, positions, :
            ].detach().cpu().float()
        handles.append(
            DECODER_LAYERS[OUTCOME_LAYER].register_forward_hook(mediator_hook)
        )
    try:
        with torch.inference_mode():
            output = adapter.model(**inputs, use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    records = score_rows(example, output.logits, positions, token_by_label)
    mediator_values = (
        outcome_probe_margins(mediator['states'])
        if capture_l50 and layer_index < OUTCOME_LAYER else [None] * len(records)
    )
    selected_counts = token_mask.sum(dim=1).detach().cpu().tolist()
    for record, dose, mediator_value, token_count in zip(
        records, doses, mediator_values, selected_counts
    ):
        record.update({
            'layer_index': int(layer_index),
            'axis_name': axis_name,
            'token_group': token_group,
            'dose_units': float(dose),
            'step_size': float(step),
            'selected_token_count': int(token_count),
            'mediator_left_right_margin': (
                None if mediator_value is None else float(mediator_value)
            ),
        })
    del inputs, output
    return records

def run_dose_series(
    example, layer_index, direction, step, doses, token_group, axis_name,
    capture_l50=True,
):
    records = []
    for dose_batch in chunks(doses, INTERVENTION_BATCH_SIZE):
        records.extend(forward_intervention_batch(
            example, layer_index, direction, step, dose_batch,
            token_group, axis_name, capture_l50=capture_l50,
        ))
    return records

print('Intervention utilities ready.')


In [ ]:
%%cell_progress 8/15 Run token-mask, zero-dose, and direction-orientation smoke tests
cell_stage('Testing all three token groups at zero dose')
smoke_example = route_dev_examples[0]
smoke_rows = []
for token_group in TOKEN_GROUPS:
    axis = axis_payload['directions']['42']['net_torque_rank1']
    smoke_rows.extend(run_dose_series(
        smoke_example, 42, axis['vector'], axis['step'], (0.0,),
        token_group, 'net_torque_rank1', capture_l50=True,
    ))
assert len(smoke_rows) == len(TOKEN_GROUPS)
zero_margins = [row['left_right_margin'] for row in smoke_rows]
assert max(zero_margins) - min(zero_margins) < 1e-5, zero_margins
assert all(np.isfinite(row['mediator_left_right_margin']) for row in smoke_rows)
display(pd.DataFrame(smoke_rows)[[
    'token_group', 'selected_token_count',
    'left_right_margin', 'mediator_left_right_margin',
]])
print('Zero-dose parity: passed')


## 3. Development-only route search

This section may choose a candidate, but it cannot read the held-out intervention directory.


In [ ]:
%%cell_progress 9/15 Run or resume the route-development candidate grid
cell_stage('Checking development-run identity')
dev_identity = {
    'schema_version': 'qwen27b-postlora-route-development-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'axis_sha256': sha256_file(AXIS_PATH),
    'manifest_sha256': sha256_file(DEV_MANIFEST_PATH),
    'candidate_layers': list(PHYSICAL_LAYERS),
    'candidate_token_groups': list(TOKEN_GROUPS),
    'candidate_axes': list(PHYSICAL_AXIS_NAMES),
    'doses': list(DEV_DOSES),
}
DEV_IDENTITY_PATH = DEV_DIR / 'identity.json'
if DEV_IDENTITY_PATH.is_file():
    assert json.loads(DEV_IDENTITY_PATH.read_text(encoding='utf-8')) == dev_identity
else:
    write_json_atomic(DEV_IDENTITY_PATH, dev_identity)

completed = {
    file_path.stem for file_path in DEV_DIR.glob('*.json')
    if file_path.name != 'identity.json'
}
for example in tqdm(
    route_dev_examples, desc='Route-development interventions', unit='prompt'
):
    safe_id = hashlib.sha256(example.example_id.encode('utf-8')).hexdigest()[:20]
    output_path = DEV_DIR / f'{safe_id}.json'
    if output_path.stem in completed:
        continue
    records = []
    for layer_index in PHYSICAL_LAYERS:
        for token_group in TOKEN_GROUPS:
            for axis_name in PHYSICAL_AXIS_NAMES:
                axis = axis_payload['directions'][str(layer_index)][axis_name]
                records.extend(run_dose_series(
                    example, layer_index, axis['vector'], axis['step'],
                    DEV_DOSES, token_group, axis_name, capture_l50=True,
                ))
    assert len(records) == (
        len(PHYSICAL_LAYERS) * len(TOKEN_GROUPS)
        * len(PHYSICAL_AXIS_NAMES) * len(DEV_DOSES)
    )
    write_json_atomic(output_path, {
        'schema_version': 'qwen27b-postlora-route-development-records-1.0',
        'example_id': example.example_id,
        'records': records,
    })
print('Development prompts complete:', len(route_dev_examples))


In [ ]:
%%cell_progress 10/15 Select and freeze one candidate without opening confirmatory interventions
cell_stage('Aggregating scene-level development slopes')
dev_records = []
for file_path in sorted(DEV_DIR.glob('*.json')):
    if file_path.name == 'identity.json':
        continue
    dev_records.extend(json.loads(file_path.read_text(encoding='utf-8'))['records'])
dev_frame = pd.DataFrame(dev_records)
assert dev_frame['example_id'].nunique() == 24

def bootstrap_summary(values, seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    assert len(values)
    rng = np.random.default_rng(seed)
    draws = np.empty(BOOTSTRAP_REPLICATES, dtype=float)
    for index in range(BOOTSTRAP_REPLICATES):
        sampled = values[rng.integers(0, len(values), size=len(values))]
        draws[index] = float(sampled.mean())
    low, high = np.quantile(draws, [0.025, 0.975])
    signs = rng.choice((-1.0, 1.0), size=(SIGN_FLIP_REPLICATES, len(values)))
    null = np.mean(signs * values[None, :], axis=1)
    p_value = float(
        (np.sum(np.abs(null) >= abs(values.mean())) + 1)
        / (SIGN_FLIP_REPLICATES + 1)
    )
    return {
        'n_scenes': int(len(values)),
        'mean': float(values.mean()),
        'ci_low': float(low),
        'ci_high': float(high),
        'sign_flip_p': p_value,
    }

dev_slope_rows = []
for keys, group in dev_frame.groupby(
    ['scene_id', 'layer_index', 'token_group', 'axis_name'], sort=True
):
    scene_id, layer_index, token_group, axis_name = keys
    group = group.sort_values('dose_units')
    dev_slope_rows.append({
        'scene_id': scene_id,
        'layer_index': int(layer_index),
        'token_group': token_group,
        'axis_name': axis_name,
        'output_slope': float(np.polyfit(
            group['dose_units'], group['left_right_margin'], 1
        )[0]),
        'mediator_slope': float(np.polyfit(
            group['dose_units'], group['mediator_left_right_margin'], 1
        )[0]),
    })
dev_slopes = pd.DataFrame(dev_slope_rows)
dev_slopes.to_csv(ANALYSIS_DIR / 'route_development_scene_slopes.csv', index=False)

candidate_rows = []
for keys, group in dev_slopes.groupby(
    ['layer_index', 'token_group', 'axis_name'], sort=True
):
    layer_index, token_group, axis_name = keys
    output_summary = bootstrap_summary(
        group['output_slope'], SEED + 1000 + len(candidate_rows)
    )
    mediator_summary = bootstrap_summary(
        group['mediator_slope'], SEED + 2000 + len(candidate_rows)
    )
    candidate_rows.append({
        'layer_index': int(layer_index),
        'token_group': token_group,
        'axis_name': axis_name,
        **{f'output_{key}': value for key, value in output_summary.items()},
        **{f'mediator_{key}': value for key, value in mediator_summary.items()},
        'output_positive_fraction': float((group['output_slope'] > 0).mean()),
    })
candidate_summary = pd.DataFrame(candidate_rows)
candidate_summary['candidate_name'] = candidate_summary.apply(
    lambda row: f"L{int(row['layer_index'])}|{row['token_group']}|{row['axis_name']}",
    axis=1,
)
candidate_summary = candidate_summary.sort_values(
    ['output_mean', 'mediator_mean', 'candidate_name'],
    ascending=[False, False, True],
).reset_index(drop=True)
candidate_summary.to_csv(ANALYSIS_DIR / 'route_development_candidates.csv', index=False)
selected = candidate_summary.iloc[0].to_dict()

SELECTION_PATH = OUTPUT_ROOT / 'FROZEN_ROUTE_SELECTION.json'
selection_payload = {
    'schema_version': 'qwen27b-postlora-frozen-route-selection-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'development_records_identity_sha256': sha256_file(DEV_IDENTITY_PATH),
    'selection_rule': protocol_identity['selection_rule'],
    'selected_candidate': selected,
    'selected_at_utc': datetime.now(timezone.utc).isoformat(),
    'confirmatory_interventions_read_before_selection': False,
}
if SELECTION_PATH.is_file():
    previous = json.loads(SELECTION_PATH.read_text(encoding='utf-8'))
    for key in ('layer_index', 'token_group', 'axis_name', 'candidate_name'):
        assert previous['selected_candidate'][key] == selected[key], (
            key, previous['selected_candidate'][key], selected[key]
        )
    selection_payload = previous
else:
    assert not any(TEST_DIR.glob('*.json')), (
        'Confirmatory records already exist before route selection was frozen.'
    )
    write_json_atomic(SELECTION_PATH, selection_payload)

selected = selection_payload['selected_candidate']
SELECTED_LAYER = int(selected['layer_index'])
SELECTED_TOKEN_GROUP = str(selected['token_group'])
SELECTED_AXIS_NAME = str(selected['axis_name'])
selected_axis = axis_payload['directions'][str(SELECTED_LAYER)][SELECTED_AXIS_NAME]

display(candidate_summary.head(10).round(5))
print('Frozen candidate:', selected['candidate_name'])


## 4. Intervention-confirmatory test

The selected direction receives a dense bidirectional dose sweep.


In [ ]:
%%cell_progress 11/15 Run or resume confirmatory torque and 32-direction controls
cell_stage('Creating frozen orthonormal random controls')
physical_matrix = np.stack([
    axis_payload['directions'][str(SELECTED_LAYER)][name]['vector'].numpy()
    for name in PHYSICAL_AXIS_NAMES
], axis=1)
physical_basis, _ = np.linalg.qr(physical_matrix)
rng = np.random.default_rng(SEED + SELECTED_LAYER)
random_vectors = []
for _ in range(RANDOM_CONTROL_COUNT):
    candidate = rng.normal(size=physical_basis.shape[0]).astype(np.float32)
    candidate -= physical_basis @ (physical_basis.T @ candidate)
    for previous in random_vectors:
        candidate -= previous * float(previous @ candidate)
    candidate = candidate / np.linalg.norm(candidate)
    random_vectors.append(candidate.astype(np.float32))
random_matrix = np.stack(random_vectors)
assert np.max(np.abs(random_matrix @ physical_basis)) < 1e-5
assert np.max(np.abs(random_matrix @ random_matrix.T - np.eye(
    RANDOM_CONTROL_COUNT
))) < 1e-4

RANDOM_AXIS_PATH = OUTPUT_ROOT / 'frozen_random_control_axes.pt'
random_payload = {
    'schema_version': 'qwen27b-postlora-random-controls-1.0',
    'selection_sha256': sha256_file(SELECTION_PATH),
    'layer_index': SELECTED_LAYER,
    'token_group': SELECTED_TOKEN_GROUP,
    'physical_axis_names_orthogonalized_against': list(PHYSICAL_AXIS_NAMES),
    'vectors': torch.from_numpy(random_matrix),
    'step': float(selected_axis['step']),
    'seed': SEED + SELECTED_LAYER,
}
if RANDOM_AXIS_PATH.is_file():
    previous = torch.load(RANDOM_AXIS_PATH, map_location='cpu', weights_only=False)
    assert torch.allclose(previous['vectors'], random_payload['vectors'], atol=1e-7, rtol=0)
else:
    torch.save(random_payload, RANDOM_AXIS_PATH)

test_identity = {
    'schema_version': 'qwen27b-postlora-intervention-confirmatory-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'selection_sha256': sha256_file(SELECTION_PATH),
    'random_axes_sha256': sha256_file(RANDOM_AXIS_PATH),
    'manifest_sha256': sha256_file(TEST_MANIFEST_PATH),
    'selected_layer': SELECTED_LAYER,
    'selected_token_group': SELECTED_TOKEN_GROUP,
    'selected_axis_name': SELECTED_AXIS_NAME,
    'torque_doses': list(TEST_DOSES),
    'random_doses': list(RANDOM_DOSES),
}
TEST_IDENTITY_PATH = TEST_DIR / 'identity.json'
if TEST_IDENTITY_PATH.is_file():
    assert json.loads(TEST_IDENTITY_PATH.read_text(encoding='utf-8')) == test_identity
else:
    write_json_atomic(TEST_IDENTITY_PATH, test_identity)

completed = {
    file_path.stem for file_path in TEST_DIR.glob('*.json')
    if file_path.name != 'identity.json'
}
for example in tqdm(
    route_test_examples, desc='Confirmatory torque/random interventions', unit='prompt'
):
    safe_id = hashlib.sha256(example.example_id.encode('utf-8')).hexdigest()[:20]
    output_path = TEST_DIR / f'{safe_id}.json'
    if output_path.stem in completed:
        continue
    records = run_dose_series(
        example, SELECTED_LAYER, selected_axis['vector'], selected_axis['step'],
        TEST_DOSES, SELECTED_TOKEN_GROUP, SELECTED_AXIS_NAME,
        capture_l50=True,
    )
    for random_index, vector in enumerate(random_matrix):
        records.extend(run_dose_series(
            example, SELECTED_LAYER, torch.from_numpy(vector),
            selected_axis['step'], RANDOM_DOSES, SELECTED_TOKEN_GROUP,
            f'random_orthogonal_{random_index:02d}', capture_l50=True,
        ))
    assert len(records) == len(TEST_DOSES) + RANDOM_CONTROL_COUNT * len(RANDOM_DOSES)
    write_json_atomic(output_path, {
        'schema_version': 'qwen27b-postlora-confirmatory-intervention-records-1.0',
        'example_id': example.example_id,
        'records': records,
    })
print('Confirmatory prompts complete:', len(route_test_examples))


In [ ]:
%%cell_progress 12/15 Run or resume the L50 outcome-direction positive control
cell_stage('Checking positive-control identity')
positive_identity = {
    'schema_version': 'qwen27b-postlora-l50-positive-control-1.0',
    'protocol_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'manifest_sha256': sha256_file(TEST_MANIFEST_PATH),
    'axis_sha256': sha256_file(AXIS_PATH),
    'layer_index': OUTCOME_LAYER,
    'token_group': 'final_token',
    'doses': list(POSITIVE_CONTROL_DOSES),
}
POSITIVE_IDENTITY_PATH = POSITIVE_CONTROL_DIR / 'identity.json'
if POSITIVE_IDENTITY_PATH.is_file():
    assert json.loads(POSITIVE_IDENTITY_PATH.read_text(encoding='utf-8')) == positive_identity
else:
    write_json_atomic(POSITIVE_IDENTITY_PATH, positive_identity)

completed = {
    file_path.stem for file_path in POSITIVE_CONTROL_DIR.glob('*.json')
    if file_path.name != 'identity.json'
}
positive_axis = axis_payload['outcome_direction']
for example in tqdm(
    route_test_examples, desc='L50 outcome-direction positive control', unit='prompt'
):
    safe_id = hashlib.sha256(example.example_id.encode('utf-8')).hexdigest()[:20]
    output_path = POSITIVE_CONTROL_DIR / f'{safe_id}.json'
    if output_path.stem in completed:
        continue
    records = run_dose_series(
        example, OUTCOME_LAYER, positive_axis['vector'], positive_axis['step'],
        POSITIVE_CONTROL_DOSES, 'final_token', 'l50_outcome_left_minus_right',
        capture_l50=False,
    )
    write_json_atomic(output_path, {
        'schema_version': 'qwen27b-postlora-positive-control-records-1.0',
        'example_id': example.example_id,
        'records': records,
    })
print('Positive-control prompts complete:', len(route_test_examples))


## 5. Frozen-gate analysis

Direct torque control requires a positive scene-level slope, a positive −4→+4 contrast, empirical superiority to the 32 random directions, and positive effects in all three gold-label strata.


In [ ]:
%%cell_progress 13/15 Analyze held-out direction control at the physical-scene level
cell_stage('Loading confirmatory intervention records')
def load_records(directory):
    rows = []
    for file_path in sorted(Path(directory).glob('*.json')):
        if file_path.name == 'identity.json':
            continue
        rows.extend(json.loads(file_path.read_text(encoding='utf-8'))['records'])
    return pd.DataFrame(rows)

test_records = load_records(TEST_DIR)
positive_records = load_records(POSITIVE_CONTROL_DIR)
assert test_records['example_id'].nunique() == len(route_test_examples)
assert positive_records['example_id'].nunique() == len(route_test_examples)

cell_stage('Averaging the three option permutations inside each scene')
numeric_columns = [
    'correct', 'left_right_margin', 'correct_margin',
    'mediator_left_right_margin',
]
test_scene = (
    test_records.assign(correct=test_records['correct'].astype(float))
    .groupby(['scene_id', 'axis_name', 'dose_units'], as_index=False)[numeric_columns]
    .mean()
)
positive_scene = (
    positive_records.assign(correct=positive_records['correct'].astype(float))
    .groupby(['scene_id', 'axis_name', 'dose_units'], as_index=False)[
        ['correct', 'left_right_margin', 'correct_margin']
    ].mean()
)
test_scene.to_csv(ANALYSIS_DIR / 'confirmatory_scene_metrics.csv', index=False)
positive_scene.to_csv(ANALYSIS_DIR / 'positive_control_scene_metrics.csv', index=False)

def scene_slopes(frame, value_column, allowed_doses=None):
    current = frame
    if allowed_doses is not None:
        current = current[current['dose_units'].isin(allowed_doses)]
    rows = []
    for (scene_id, axis_name), group in current.groupby(
        ['scene_id', 'axis_name'], sort=True
    ):
        rows.append({
            'scene_id': scene_id,
            'axis_name': axis_name,
            'slope': float(np.polyfit(
                group['dose_units'], group[value_column], 1
            )[0]),
        })
    return pd.DataFrame(rows)

output_slopes = scene_slopes(
    test_scene, 'left_right_margin', PRIMARY_TEST_DOSES
)
mediator_slopes = scene_slopes(
    test_scene, 'mediator_left_right_margin', PRIMARY_TEST_DOSES
)
positive_slopes = scene_slopes(
    positive_scene, 'left_right_margin', POSITIVE_CONTROL_DOSES
)

torque_output_values = output_slopes.loc[
    output_slopes['axis_name'] == SELECTED_AXIS_NAME, 'slope'
].to_numpy()
torque_mediator_values = mediator_slopes.loc[
    mediator_slopes['axis_name'] == SELECTED_AXIS_NAME, 'slope'
].to_numpy()
positive_values = positive_slopes['slope'].to_numpy()
torque_output = bootstrap_summary(torque_output_values, SEED + 4001)
torque_mediator = bootstrap_summary(torque_mediator_values, SEED + 4002)
positive_control = bootstrap_summary(positive_values, SEED + 4003)

random_output_means = (
    output_slopes[output_slopes['axis_name'].str.startswith('random_orthogonal_')]
    .groupby('axis_name')['slope'].mean().sort_index()
)
random_mediator_means = (
    mediator_slopes[mediator_slopes['axis_name'].str.startswith('random_orthogonal_')]
    .groupby('axis_name')['slope'].mean().sort_index()
)
assert len(random_output_means) == len(random_mediator_means) == RANDOM_CONTROL_COUNT
output_empirical_p = float(
    (1 + np.sum(random_output_means.to_numpy() >= torque_output['mean']))
    / (RANDOM_CONTROL_COUNT + 1)
)
mediator_empirical_p = float(
    (1 + np.sum(random_mediator_means.to_numpy() >= torque_mediator['mean']))
    / (RANDOM_CONTROL_COUNT + 1)
)

selected_scene = test_scene[test_scene['axis_name'] == SELECTED_AXIS_NAME]
endpoint = selected_scene.pivot(
    index='scene_id', columns='dose_units', values='left_right_margin'
)
endpoint_effect = bootstrap_summary(
    endpoint[4.0] - endpoint[-4.0], SEED + 4004
)

cell_stage('Testing A/B/C permutation robustness without averaging')
permutation_rows = []
selected_prompt = test_records[
    (test_records['axis_name'] == SELECTED_AXIS_NAME)
    & test_records['dose_units'].isin(PRIMARY_TEST_DOSES)
]
for (scene_id, gold_label), group in selected_prompt.groupby(
    ['scene_id', 'gold_label'], sort=True
):
    permutation_rows.append({
        'scene_id': scene_id,
        'gold_label': gold_label,
        'slope': float(np.polyfit(
            group['dose_units'], group['left_right_margin'], 1
        )[0]),
    })
permutation_slopes = pd.DataFrame(permutation_rows)
permutation_summaries = {}
for label_index, gold_label in enumerate(('A', 'B', 'C')):
    permutation_summaries[gold_label] = bootstrap_summary(
        permutation_slopes.loc[
            permutation_slopes['gold_label'] == gold_label, 'slope'
        ],
        SEED + 4100 + label_index,
    )

dose_means = selected_scene.groupby('dose_units')['left_right_margin'].mean()
primary_means = dose_means.reindex(PRIMARY_TEST_DOSES)
aggregate_spearman = float(spearmanr(
    np.asarray(PRIMARY_TEST_DOSES), primary_means.to_numpy()
).statistic)
mediator_output_correlation = float(np.corrcoef(
    torque_output_values, torque_mediator_values
)[0, 1])

gates = {
    'l50_outcome_direction_positive_control': positive_control['ci_low'] > 0,
    'selected_torque_output_positive': torque_output['ci_low'] > 0,
    'selected_torque_beats_32_random_directions': output_empirical_p <= ALPHA_LEVEL,
    'selected_torque_bidirectional_endpoint': endpoint_effect['ci_low'] > 0,
    'selected_torque_permutation_robust': all(
        permutation_summaries[label]['ci_low'] > 0 for label in ('A', 'B', 'C')
    ),
    'selected_torque_reaches_l50': (
        torque_mediator['ci_low'] > 0
        and mediator_empirical_p <= ALPHA_LEVEL
    ),
}
gates['direct_torque_control_confirmed'] = all([
    gates['l50_outcome_direction_positive_control'],
    gates['selected_torque_output_positive'],
    gates['selected_torque_beats_32_random_directions'],
    gates['selected_torque_bidirectional_endpoint'],
    gates['selected_torque_permutation_robust'],
])

decision = {
    'schema_version': 'qwen27b-postlora-torque-control-decision-1.0',
    'selected_candidate': selected,
    'torque_output_slope': torque_output,
    'torque_mediator_slope': torque_mediator,
    'endpoint_plus4_minus_minus4': endpoint_effect,
    'l50_positive_control_output_slope': positive_control,
    'random_control_output_mean_slopes': random_output_means.to_dict(),
    'random_control_mediator_mean_slopes': random_mediator_means.to_dict(),
    'output_empirical_p_vs_32_random': output_empirical_p,
    'mediator_empirical_p_vs_32_random': mediator_empirical_p,
    'permutation_output_slopes': permutation_summaries,
    'aggregate_dose_spearman': aggregate_spearman,
    'scene_output_mediator_slope_correlation': mediator_output_correlation,
    'gates': gates,
    'interpretation_tree': {
        'direct_torque_control_confirmed': (
            'A probe-train-defined physical direction has held-out, '
            'permutation-robust, random-control-specific causal leverage.'
        ),
        'positive_control_only': (
            'The intervention can control the late readout, but the tested '
            'physical direction/site is not the causal answer-control variable.'
        ),
        'positive_control_fails': (
            'The continuous steering calibration is not validated; do not use '
            'this run to reject a torque mechanism.'
        ),
    },
    'claim_limit': (
        'Passing establishes directional control under this hybrid intervention; '
        'it does not identify a unique circuit or prove explicit multiplication.'
    ),
}
DECISION_PATH = ANALYSIS_DIR / 'torque_control_decision.json'
write_json_atomic(DECISION_PATH, decision)
permutation_slopes.to_csv(ANALYSIS_DIR / 'permutation_scene_slopes.csv', index=False)
output_slopes.to_csv(ANALYSIS_DIR / 'output_scene_slopes.csv', index=False)
mediator_slopes.to_csv(ANALYSIS_DIR / 'mediator_scene_slopes.csv', index=False)

display(pd.DataFrame({
    'quantity': [
        'selected torque output slope', 'selected torque L50 slope',
        '+4 minus -4 output contrast', 'L50 outcome-axis output slope',
    ],
    'mean': [
        torque_output['mean'], torque_mediator['mean'],
        endpoint_effect['mean'], positive_control['mean'],
    ],
    'ci_low': [
        torque_output['ci_low'], torque_mediator['ci_low'],
        endpoint_effect['ci_low'], positive_control['ci_low'],
    ],
    'ci_high': [
        torque_output['ci_high'], torque_mediator['ci_high'],
        endpoint_effect['ci_high'], positive_control['ci_high'],
    ],
}).round(5))
print('Output empirical p vs 32 random:', output_empirical_p)
print('Mediator empirical p vs 32 random:', mediator_empirical_p)
print('Permutation summaries:', permutation_summaries)
print('Frozen gates:', gates)


In [ ]:
%%cell_progress 14/15 Render the preregistered summary figure
cell_stage('Rendering development and held-out diagnostics')
plt.style.use('seaborn-v0_8-whitegrid')
figure, axes = plt.subplots(2, 2, figsize=(15, 11))

plot_dev = candidate_summary.copy()
for axis_name, color in (
    ('net_torque_rank1', '#4C78A8'),
    ('side_torque_rank2', '#F58518'),
):
    current = plot_dev[plot_dev['axis_name'] == axis_name]
    axes[0, 0].scatter(
        current['mediator_mean'], current['output_mean'],
        label=axis_name, color=color, s=55, alpha=0.8,
    )
axes[0, 0].axhline(0, color='black', linewidth=1)
axes[0, 0].axvline(0, color='black', linewidth=1)
axes[0, 0].set_xlabel('Route-dev L50 mediator slope')
axes[0, 0].set_ylabel('Route-dev output slope')
axes[0, 0].set_title('A. Candidate search (development only)')
axes[0, 0].legend()

selected_means = selected_scene.groupby('dose_units', as_index=False).agg(
    output=('left_right_margin', 'mean'),
    mediator=('mediator_left_right_margin', 'mean'),
)
axes[0, 1].plot(
    selected_means['dose_units'], selected_means['output'],
    marker='o', color='#4C78A8', label='selected physical direction',
)
random_scene = test_scene[
    test_scene['axis_name'].str.startswith('random_orthogonal_')
]
random_by_axis_dose = random_scene.groupby(
    ['axis_name', 'dose_units'], as_index=False
)['left_right_margin'].mean()
random_band = random_by_axis_dose.groupby('dose_units')['left_right_margin'].agg(
    low=lambda values: np.quantile(values, 0.05),
    median='median',
    high=lambda values: np.quantile(values, 0.95),
).reset_index()
axes[0, 1].fill_between(
    random_band['dose_units'], random_band['low'], random_band['high'],
    color='#BAB0AC', alpha=0.35, label='random 5–95% band',
)
axes[0, 1].plot(
    random_band['dose_units'], random_band['median'],
    color='#79706E', linestyle='--', label='random median',
)
axes[0, 1].set_xlabel('Dose (projection-SD units)')
axes[0, 1].set_ylabel('Left minus right output margin')
axes[0, 1].set_title('B. Held-out output control')
axes[0, 1].legend(fontsize=8)

axes[1, 0].plot(
    selected_means['dose_units'], selected_means['mediator'],
    marker='o', color='#54A24B', label='selected physical direction',
)
random_mediator_band = random_scene.groupby(
    ['axis_name', 'dose_units'], as_index=False
)['mediator_left_right_margin'].mean().groupby('dose_units')[
    'mediator_left_right_margin'
].agg(
    low=lambda values: np.quantile(values, 0.05),
    median='median',
    high=lambda values: np.quantile(values, 0.95),
).reset_index()
axes[1, 0].fill_between(
    random_mediator_band['dose_units'], random_mediator_band['low'],
    random_mediator_band['high'], color='#BAB0AC', alpha=0.35,
    label='random 5–95% band',
)
axes[1, 0].set_xlabel('Dose (projection-SD units)')
axes[1, 0].set_ylabel('Frozen L50 outcome-probe margin')
axes[1, 0].set_title('C. Held-out mediator transport')
axes[1, 0].legend(fontsize=8)

positive_means = positive_scene.groupby('dose_units', as_index=False)[
    'left_right_margin'
].mean()
axes[1, 1].plot(
    positive_means['dose_units'], positive_means['left_right_margin'],
    marker='o', color='#E45756', label='L50 outcome direction',
)
axes[1, 1].axhline(0, color='black', linewidth=1)
axes[1, 1].set_xlabel('Dose (projection-SD units)')
axes[1, 1].set_ylabel('Left minus right output margin')
axes[1, 1].set_title('D. Continuous-steering positive control')
axes[1, 1].legend()

plt.tight_layout()
SUMMARY_FIGURE_PATH = FIGURE_DIR / 'torque_control_stresstest_summary.png'
plt.savefig(SUMMARY_FIGURE_PATH, dpi=180, bbox_inches='tight')
plt.show()


In [ ]:
%%cell_progress 15/15 Verify the complete stress-test package
cell_stage('Checking all required outputs and invariants')
required_outputs = (
    PROTOCOL_LOCK_PATH, AXIS_PATH,
    ANALYSIS_DIR / 'frozen_axis_metrics.csv',
    DEV_MANIFEST_PATH, TEST_MANIFEST_PATH,
    DEV_IDENTITY_PATH, SELECTION_PATH,
    RANDOM_AXIS_PATH, TEST_IDENTITY_PATH, POSITIVE_IDENTITY_PATH,
    ANALYSIS_DIR / 'route_development_scene_slopes.csv',
    ANALYSIS_DIR / 'route_development_candidates.csv',
    ANALYSIS_DIR / 'confirmatory_scene_metrics.csv',
    ANALYSIS_DIR / 'positive_control_scene_metrics.csv',
    ANALYSIS_DIR / 'permutation_scene_slopes.csv',
    ANALYSIS_DIR / 'output_scene_slopes.csv',
    ANALYSIS_DIR / 'mediator_scene_slopes.csv',
    DECISION_PATH, SUMMARY_FIGURE_PATH,
)
missing = [str(path) for path in required_outputs if not Path(path).is_file()]
assert not missing, 'Missing required outputs:\n' + '\n'.join(missing)
assert len([
    path for path in DEV_DIR.glob('*.json') if path.name != 'identity.json'
]) == len(route_dev_examples)
assert len([
    path for path in TEST_DIR.glob('*.json') if path.name != 'identity.json'
]) == len(route_test_examples)
assert len([
    path for path in POSITIVE_CONTROL_DIR.glob('*.json')
    if path.name != 'identity.json'
]) == len(route_test_examples)

completion = {
    'schema_version': 'qwen27b-postlora-torque-control-stresstest-complete-1.0',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'adapter_dir': str(ADAPTER_DIR),
    'route_development_scenes': len(route_dev_scene_ids),
    'intervention_confirmatory_scenes': len(route_test_scene_ids),
    'confirmatory_prompts': len(route_test_examples),
    'random_control_directions': RANDOM_CONTROL_COUNT,
    'selected_candidate': selected,
    'decision_path': str(DECISION_PATH),
    'summary_figure_path': str(SUMMARY_FIGURE_PATH),
    'required_outputs': [str(path) for path in required_outputs],
}
RUN_COMPLETE_PATH = OUTPUT_ROOT / 'RUN_COMPLETE.json'
write_json_atomic(RUN_COMPLETE_PATH, completion)
print('Torque-control stress test complete:', RUN_COMPLETE_PATH)
print('Primary decision:', DECISION_PATH)
print('Summary figure:', SUMMARY_FIGURE_PATH)
